# Reconnaissance de panneaux de signalisation avec un CNN
**Projet GTSRB · TensorFlow/Keras · Google Colab · 43 classes**

Objectif : reconnaître la catégorie d'un panneau déjà cadré dans une image.
Ce modèle réalise une **classification** ; pour une photo de rue contenant plusieurs objets,
il faut d'abord détecter et recadrer le panneau.

### Démarrage dans Colab
1. Importer ce fichier `.ipynb` via **Fichier → Importer un notebook**.
2. Sélectionner **Exécution → Modifier le type d'exécution → GPU** si disponible.
3. Exécuter les cellules dans l'ordre. Le dataset public est téléchargé depuis Kaggle.
4. Télécharger l'archive finale pour conserver le modèle et les résultats avant la fin de la session.

Dataset : [GTSRB sur Kaggle](https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign).
Le benchmark contient 39 209 images d'entraînement et 12 630 images de test, soit 51 839 images.
Les effectifs et les 43 classes seront vérifiés sur les CSV téléchargés.
Les scores de ce notebook seront calculés à l'exécution : aucun résultat n'est prérempli.


## 1. Dépendances et configuration
Colab fournit déjà TensorFlow. On conserve sa version pour rester compatible avec son GPU.
En local : Python 3.11 ou 3.12 et `pip install -r requirements.txt`.


In [ ]:
%pip -q install "kagglehub>=0.4,<1" "scikit-learn>=1.4,<2" "Pillow>=10,<13" "pandas>=2,<4" "matplotlib>=3.8,<4"


In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import sys
import json
import platform
import shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from IPython.display import display
import kagglehub
from importlib.metadata import version

SEED = 42
IMG_SIZE = 48
BATCH_SIZE = 64
EPOCHS = 20
NUM_CLASSES = 43
SMOKE_TEST = False  # True : quelques images/classe, 1 époque ; scores non représentatifs.
RUN_UPLOAD_DEMO = False  # True : ouvre le sélecteur d'image personnel dans Colab.
DATASET_HANDLE = "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/versions/1"
LOCAL_DATA_DIR = ""  # Option : chemin vers un dataset Kaggle déjà extrait.
OUTPUT_DIR = Path("artifacts/smoke" if SMOKE_TEST else "artifacts/full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
keras.utils.set_random_seed(SEED)
print("Python :", platform.python_version(), "| TensorFlow :", tf.__version__)
print("GPU :", tf.config.list_physical_devices("GPU"))
print("Mode :", "VÉRIFICATION RAPIDE" if SMOKE_TEST else "ENTRAÎNEMENT COMPLET")


## 2. Télécharger et vérifier les données Kaggle
Le téléchargement public avec `kagglehub` ne demande généralement pas d'authentification.
Si Kaggle la demande, exécuter `kagglehub.login()` dans une cellule séparée puis relancer.
Ne jamais coller une clé dans le notebook ou GitHub. Autre possibilité : télécharger le ZIP
depuis Kaggle, l'extraire, puis renseigner `LOCAL_DATA_DIR` dans la configuration.
Le téléchargement peut prendre plusieurs minutes et nécessite de l'espace disque.


In [ ]:
if LOCAL_DATA_DIR:
    dataset_dir = Path(LOCAL_DATA_DIR).expanduser().resolve()
else:
    dataset_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))

def locate_csv(root, filename):
    matches = sorted(p for p in root.rglob("*.csv") if p.name.lower() == filename.lower())
    if len(matches) != 1:
        raise FileNotFoundError(f"Un seul {filename} attendu dans {root}, trouvé : {matches}")
    return matches[0]

train_csv = locate_csv(dataset_dir, "Train.csv")
test_csv = locate_csv(dataset_dir, "Test.csv")
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

def prepare_metadata(frame, csv_path):
    frame = frame.copy()
    required = {"Path", "ClassId"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Colonnes manquantes : {required - set(frame.columns)}")
    frame["ClassId"] = pd.to_numeric(frame["ClassId"], errors="raise").astype(int)
    if not frame["ClassId"].between(0, NUM_CLASSES - 1).all():
        raise ValueError("Les classes doivent être comprises entre 0 et 42.")
    def resolve_image(relative):
        relative = Path(str(relative).replace(chr(92), "/"))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Chemin invalide : {relative}")
        candidate = csv_path.parent / relative
        if not candidate.is_file():
            raise FileNotFoundError(candidate)
        return str(candidate.resolve())
    frame["filepath"] = frame["Path"].map(resolve_image)
    assert not frame["filepath"].duplicated().any(), "Images dupliquées dans le CSV."
    return frame

train_df = prepare_metadata(train_df, train_csv)
test_df = prepare_metadata(test_df, test_csv)
assert len(train_df) == 39209 and len(test_df) == 12630, "Version du dataset inattendue."
assert set(train_df.ClassId) == set(range(NUM_CLASSES))
assert set(test_df.ClassId) == set(range(NUM_CLASSES))
assert set(train_df.filepath).isdisjoint(test_df.filepath)
print(f"Train officiel : {len(train_df):,} | Test officiel : {len(test_df):,}")
display(train_df.head())


In [ ]:
CLASS_NAMES = [
    "Limitation à 20 km/h", "Limitation à 30 km/h", "Limitation à 50 km/h",
    "Limitation à 60 km/h", "Limitation à 70 km/h", "Limitation à 80 km/h",
    "Fin de limitation à 80 km/h", "Limitation à 100 km/h", "Limitation à 120 km/h",
    "Interdiction de dépasser", "Dépassement interdit aux véhicules de plus de 3,5 t",
    "Priorité à la prochaine intersection", "Route prioritaire", "Cédez le passage",
    "Stop", "Circulation interdite", "Véhicules de plus de 3,5 t interdits",
    "Sens interdit", "Danger", "Virage dangereux à gauche", "Virage dangereux à droite",
    "Succession de virages", "Chaussée déformée", "Chaussée glissante",
    "Chaussée rétrécie à droite", "Travaux", "Feux tricolores", "Passage de piétons",
    "Enfants", "Cyclistes", "Risque de verglas ou de neige", "Passage d'animaux sauvages",
    "Fin de toutes les interdictions", "Tourner à droite", "Tourner à gauche",
    "Tout droit", "Tout droit ou à droite", "Tout droit ou à gauche",
    "Contournement par la droite", "Contournement par la gauche", "Rond-point",
    "Fin d'interdiction de dépasser", "Fin d'interdiction de dépasser pour les plus de 3,5 t",
]
assert len(CLASS_NAMES) == NUM_CLASSES

fig, ax = plt.subplots(figsize=(14, 4))
train_df.ClassId.value_counts().sort_index().plot.bar(ax=ax, color="#247a9c")
ax.set(title="Répartition des 43 classes — entraînement officiel",
       xlabel="Identifiant de classe", ylabel="Nombre d'images")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(4, 6, figsize=(12, 8))
for ax, class_id in zip(axes.flat, range(24)):
    row = train_df[train_df.ClassId == class_id].sample(1, random_state=SEED).iloc[0]
    with Image.open(row.filepath) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"{class_id} : {CLASS_NAMES[class_id]}", fontsize=8)
    ax.axis("off")
fig.tight_layout()
plt.show()


## 3. Créer une validation sans mélanger les séquences
GTSRB contient plusieurs vues d'un même panneau. Les noms d'images d'entraînement suivent
la forme `classe_séquence_image.png`. On garde une séquence entière dans un seul ensemble
avec `StratifiedGroupKFold`. Le premier des cinq plis sert de validation (environ 20 %).
L'équilibre par classe est approximatif, car les groupes sont indivisibles.

Le test officiel reste à part : il ne sert ni au choix des paramètres ni à l'arrêt anticipé.
Le mode rapide sous-échantillonne les ensembles **après** leur séparation et ne produit pas un score benchmark.


In [ ]:
stems = train_df["filepath"].map(lambda p: Path(p).stem)
assert stems.str.match(r"^\d+_\d+_\d+$").all(), "Format de séquence inattendu."
train_df["group"] = train_df["ClassId"].astype(str) + "_" + stems.str.rsplit("_", n=1).str[0]
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(splitter.split(train_df, train_df.ClassId, groups=train_df.group))
fit_df = train_df.iloc[train_idx].copy()
val_df = train_df.iloc[val_idx].copy()
assert set(fit_df.group).isdisjoint(val_df.group)
for frame in (fit_df, val_df):
    assert set(frame.ClassId) == set(range(NUM_CLASSES)), "Une classe manque dans un ensemble."

def sample_per_class(frame, count):
    return pd.concat([part.sample(min(count, len(part)), random_state=SEED)
                      for _, part in frame.groupby("ClassId")], ignore_index=True)

if SMOKE_TEST:
    fit_df = sample_per_class(fit_df, 8)
    val_df = sample_per_class(val_df, 4)
    test_df = sample_per_class(test_df, 4)

split_manifest = pd.concat([
    fit_df.assign(split="train"), val_df.assign(split="validation"),
    test_df.assign(split="test")], ignore_index=True)
split_manifest[["Path", "ClassId", "split"]].to_csv(OUTPUT_DIR / "splits.csv", index=False)
display(pd.DataFrame({
    "entraînement": fit_df.ClassId.value_counts(),
    "validation": val_df.ClassId.value_counts(),
    "test": test_df.ClassId.value_counts(),
}).sort_index())
print("Effectifs :", len(fit_df), len(val_df), len(test_df))


## 4. Pipeline d'images
Les images sont lues progressivement, converties en RGB et redimensionnées à 48 × 48.
La normalisation `[0, 255] → [0, 1]` fait partie du modèle et sera donc conservée à l'export.
On ne retourne pas les panneaux horizontalement ou verticalement : cela pourrait changer leur sens.


In [ ]:
def decode_image(path, label):
    image = tf.io.decode_png(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], antialias=True)
    image = tf.ensure_shape(image, [IMG_SIZE, IMG_SIZE, 3])
    return image, tf.cast(label, tf.int32)

def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.filepath.to_numpy(),
                                            frame.ClassId.to_numpy(dtype=np.int32)))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(fit_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)  # Ordre fixe : correspond aux lignes de test_df.
images, labels = next(iter(train_ds))
print("Batch :", images.shape, "| Labels :", labels.shape)


## 5. Construire le CNN
Trois blocs convolutifs apprennent les formes, couleurs et détails des panneaux.
Batch normalization stabilise l'apprentissage ; dropout réduit le surapprentissage.
Une augmentation légère s'applique uniquement pendant l'entraînement.
La sortie softmax contient 43 probabilités et la perte utilise des étiquettes entières.


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
augmentation = keras.Sequential([
    keras.layers.RandomRotation(0.03, fill_mode="nearest", seed=SEED),
    keras.layers.RandomTranslation(0.08, 0.08, fill_mode="nearest", seed=SEED + 1),
    keras.layers.RandomZoom(0.1, fill_mode="nearest", seed=SEED + 2),
], name="augmentation")

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_rgb_0_255")
x = keras.layers.Rescaling(1.0 / 255)(inputs)
x = augmentation(x)
for filters in (32, 64, 128):
    for _ in range(2):
        x = keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation("relu")(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Flatten()(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.4)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation="softmax", name="probabilities")(x)
model = keras.Model(inputs, outputs, name="gtsrb_cnn")
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


## 6. Entraîner et sélectionner le meilleur modèle
La meilleure époque est choisie sur la **perte de validation**.
L'arrêt anticipé limite le surapprentissage ; le taux d'apprentissage diminue si la validation stagne.
Un GPU est recommandé pour l'entraînement complet. La durée et les scores dépendent de la session.


In [ ]:
checkpoint_path = OUTPUT_DIR / "best_model.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(str(checkpoint_path), monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    keras.callbacks.CSVLogger(str(OUTPUT_DIR / "training_log.csv")),
    keras.callbacks.TerminateOnNaN(),
]
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=1 if SMOKE_TEST else EPOCHS, callbacks=callbacks)
history_df = pd.DataFrame(history.history)
history_df.to_csv(OUTPUT_DIR / "history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epoch_axis = np.arange(1, len(history_df) + 1)
for ax, metric, title in zip(axes, ["accuracy", "loss"], ["Exactitude", "Perte"]):
    ax.plot(epoch_axis, history_df[metric], marker="o", label="Entraînement")
    ax.plot(epoch_axis, history_df["val_" + metric], marker="o", label="Validation")
    ax.set(title=title, xlabel="Époque")
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "learning_curves.png", dpi=150)
plt.show()


## 7. Évaluation finale sur le test officiel
On recharge le meilleur checkpoint avant de calculer l'accuracy et le F1 macro.
Le F1 macro donne le même poids à chaque classe, utile lorsque les classes sont déséquilibrées.
La matrice de confusion est normalisée par vraie classe.
Si vous ajustez ensuite le modèle à partir de ces résultats, le test ne constitue plus une évaluation indépendante.


In [ ]:
model = keras.models.load_model(checkpoint_path)
probabilities = model.predict(test_ds, verbose=1)
y_true = test_df.ClassId.to_numpy()
y_pred = probabilities.argmax(axis=1)
assert probabilities.shape == (len(test_df), NUM_CLASSES)
assert np.isfinite(probabilities).all()
assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
metrics = {
    "mode": "smoke_test" if SMOKE_TEST else "full",
    "test_images": int(len(test_df)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                              average="macro", zero_division=0)),
    "test_loss": float(keras.losses.sparse_categorical_crossentropy(y_true, probabilities).numpy().mean()),
    "best_epoch": int(np.argmin(history_df.val_loss.to_numpy()) + 1),
}
print(json.dumps(metrics, indent=2, ensure_ascii=False))
report = classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")
display(report_df)

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
pd.DataFrame(cm, index=range(NUM_CLASSES), columns=range(NUM_CLASSES)).to_csv(OUTPUT_DIR / "confusion_matrix.csv")
cm_normalized = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(16, 14))
ConfusionMatrixDisplay(cm_normalized, display_labels=range(NUM_CLASSES)).plot(
    ax=ax, cmap="Blues", include_values=False, colorbar=True)
ax.set(title="Matrice de confusion normalisée (IDs 0–42)",
       xlabel="Classe prédite", ylabel="Classe réelle")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

predictions = test_df[["Path", "ClassId"]].copy()
predictions["PredictedClassId"] = y_pred
predictions["Confidence"] = probabilities.max(axis=1)
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)


In [ ]:
wrong = np.flatnonzero(y_true != y_pred)
if len(wrong):
    chosen = np.random.default_rng(SEED).choice(wrong, size=min(12, len(wrong)), replace=False)
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax in axes.flat:
        ax.axis("off")
    for ax, idx in zip(axes.flat, chosen):
        with Image.open(test_df.iloc[idx].filepath) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(f"Vrai : {y_true[idx]} | Prédit : {y_pred[idx]}\n"
                     f"Score softmax : {probabilities[idx].max():.1%}", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "prediction_errors.png", dpi=150)
    plt.show()
else:
    print("Aucune erreur sur cet ensemble.")


## 8. Exporter le modèle, les labels et les résultats
Le format `.keras` conserve la normalisation, l'architecture et les poids.
Le manifeste conserve le split, la version du dataset et les dépendances.
Le dossier `artifacts/` est ignoré par Git pour éviter de publier des fichiers volumineux.
Le téléchargement Colab est facultatif et s'effectue dans la cellule suivante.


In [ ]:
model.save(OUTPUT_DIR / "traffic_sign_model.keras")
(OUTPUT_DIR / "class_names.json").write_text(
    json.dumps(dict(enumerate(CLASS_NAMES)), ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": DATASET_HANDLE, "seed": SEED, "image_size": IMG_SIZE,
    "input": "RGB, float32, pixels 0..255, redimensionnement bilinéaire avec antialias",
    "normalization": "incluse dans le modèle",
    "split": "premier pli de StratifiedGroupKFold(5, shuffle=True, random_state=42), groupé par séquence",
    "samples": {"train": len(fit_df), "validation": len(val_df), "test": len(test_df)},
    "smoke_test": SMOKE_TEST,
    "versions": {package: version(package) for package in
                 ["keras", "kagglehub", "scikit-learn", "pandas", "numpy", "Pillow"]},
}
metadata["versions"]["tensorflow"] = tf.__version__
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
archive_path = shutil.make_archive(str(OUTPUT_DIR.parent / ("gtsrb_smoke" if SMOKE_TEST else "gtsrb_results")),
                                   "zip", root_dir=OUTPUT_DIR)
print("Archive créée :", archive_path)


In [ ]:
DOWNLOAD_RESULTS = False  # Passer à True puis exécuter pour télécharger l'archive dans Colab.
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print("Hors Colab, récupérer l'archive à cet emplacement :", archive_path)
    else:
        files.download(archive_path)


## 9. Prédire sur une image personnelle
L'image doit contenir **un seul panneau bien cadré**, appartenant à une des 43 classes.
La fonction applique le même redimensionnement que l'entraînement et propose les trois classes principales.
Le score softmax n'est pas une garantie de fiabilité : le modèle choisit une classe même si l'image ne contient aucun panneau connu.
Ce projet pédagogique ne doit pas être utilisé pour prendre des décisions de conduite.


In [ ]:
def predict_image(image_path, top_k=3):
    if not 1 <= top_k <= NUM_CLASSES:
        raise ValueError("top_k doit être compris entre 1 et 43.")
    with Image.open(image_path) as image:
        rgb = np.asarray(ImageOps.exif_transpose(image).convert("RGB"), dtype=np.float32)
    resized = tf.image.resize(rgb, [IMG_SIZE, IMG_SIZE], antialias=True)
    scores = model.predict(resized[None, ...], verbose=0)[0]
    top_ids = np.argsort(scores)[::-1][:top_k]
    result = pd.DataFrame({"Classe": top_ids,
                           "Panneau": [CLASS_NAMES[i] for i in top_ids],
                           "Score softmax": scores[top_ids]})
    plt.figure(figsize=(4, 4))
    plt.imshow(rgb.astype(np.uint8))
    plt.axis("off")
    plt.show()
    return result

# Démonstration automatique sur une image du test.
display(predict_image(test_df.iloc[0].filepath))

if RUN_UPLOAD_DEMO:
    try:
        from google.colab import files
    except ImportError:
        print("Hors Colab : appeler predict_image('chemin/vers/image.png').")
    else:
        uploaded = files.upload()
        for filename in uploaded:
            try:
                display(predict_image(filename))
            except (OSError, ValueError) as error:
                print(f"Image non lisible : {filename} ({error})")


## 10. Interpréter et améliorer le projet
- Comparer les courbes pour repérer le surapprentissage.
- Examiner les classes dont le rappel est faible et les confusions entre limitations de vitesse.
- Choisir les améliorations à partir de la validation : taille d'image, régularisation, nombre d'époques.
- Tester ensuite le modèle sur des conditions différentes : éclairage, flou, panneaux locaux.
- Pour reconnaître plusieurs panneaux dans une scène, ajouter un modèle de détection et un dataset adapté.

### Sources
- [Dataset Kaggle GTSRB](https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign) (version 1).
- [Benchmark original GTSRB](https://benchmark.ini.rub.de/gtsrb_dataset.html).
- [KaggleHub officiel](https://github.com/Kaggle/kagglehub) : téléchargement et authentification.
- [TensorFlow : augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation).
- [Scikit-learn : StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html).

Le code du projet est sous licence MIT. Les données restent soumises aux conditions de leur source ;
consulter la fiche Kaggle et le benchmark original pour leur utilisation et citation.
